In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go


In [ ]:

#gdf_rinde = gpd.read_file("datos_cosecha_300puntos.csv")
#gdf_rinde.head()

#Problema resuelto, Geopandas no lee .csv. Se lo lee con Pandas y luego se arma la geometría con gpd    
df = pd.read_csv('datos_cosecha_300puntos.csv')

gdf_rinde = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df['Longitud'], df['Latitud'])
)
# 3. Le asignamos el sistema de origen (Lat/Lon estándar)
gdf_rinde = gdf_rinde.set_crs(epsg=4326)

# 4. Ahora sí, podés reproyectar sin que explote nada:
gdf_rinde_metros = gdf_rinde.to_crs(gdf_rinde.estimate_utm_crs())

# 5. Reproyectamos
gdf_rp = gdf_rinde_metros.to_crs(epsg=32720)
gdf_rp.head()

# 6. Estimar superficie del perimetro
gdf_rp['superficie_m2'] = gdf_rp.geometry.buffer(10).area
gdf_rp.head()

In [ ]:
gdf_rinde['cuantiles'] = pd.qcut(gdf_rinde['Rinde_kgHa'], 5, labels=False)

gdf_rinde.plot(column='cuantiles', cmap='RdYlGn', legend=True)
plt.title('Rinde en 4 cuantiles')
plt.show()


In [ ]:
#Añadir la capa de rendimientos en un mapa interactivo con Plotly
fig = go.Figure()
fig.add_trace(go.Scattermapbox(
    lat=gdf_rinde['Latitud'],
    lon=gdf_rinde['Longitud'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=10,
        color=gdf_rinde['Rinde_kgHa'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='Rinde (kg/ha)')
    ),
    text=gdf_rinde['Rinde_kgHa'],
    hoverinfo='text'
))
fig.update_layout(
    mapbox_style="white-bg", # Ponemos el fondo en blanco
    mapbox_layers=[
        {
            "below": 'traces', # Para que el satélite quede por debajo de tus puntos de rinde
            "sourcetype": "raster",
            "sourceattribution": "Google",
            "source": [
                "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}" # URL del satélite de Google
            ]
        }
    ],
    mapbox_center={"lat": gdf_rinde['Latitud'].mean(), "lon": gdf_rinde['Longitud'].mean()},
    mapbox_zoom=13, # Te subí un poco el zoom (14 o 15 suele ser mejor para ver lotes)
    title="Rinde en el campo"
)
fig.show()

#lo hago html
fig.write_html("mapa_rinde.html")
print("Mapa guardado como mapa_rinde.html")
